In [1]:
import pandas as pd
from zeep import Client, Settings, Transport
import hashlib
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from requests.exceptions import ConnectionError, HTTPError, ChunkedEncodingError, ReadTimeout, RequestException
from urllib3.exceptions import IncompleteRead
import requests

def create_session_with_retries(retries, backoff_factor, status_forcelist):
    """Create a requests session with retries"""
    session = requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def seq_by_brenda(ec, organism, email, password, retries=5, delay=5):
    """Retrieve sequences from BRENDA given EC number and organism."""
    password = hashlib.sha256(password.encode("utf-8")).hexdigest()
    wsdl = "https://www.brenda-enzymes.org/soap/brenda_zeep.wsdl"
    settings = Settings(strict=False, xml_huge_tree=True)

    attempts = 0
    while attempts < retries:
        try:
            session = create_session_with_retries(retries, backoff_factor=delay, status_forcelist=[500, 502, 503, 504])
            transport = Transport(session=session, timeout=300)
            client = Client(wsdl, settings=settings, transport=transport)

            parameters = (email, password, "ecNumber*%s" % ec, "organism*%s" % organism, "sequence*", "noOfAminoAcids*", "firstAccessionCode*", "source*Swiss-Prot", "id*")
            entries = client.service.getSequence(*parameters)
            
            sequences = []
            if entries:
                for entry in entries:
                    sequences.append(entry['sequence'])
            return sequences
        except (ConnectionError, HTTPError, ChunkedEncodingError, ReadTimeout, IncompleteRead, RequestException) as e:
            print(f"Attempt {attempts + 1} failed with error: {e}. Retrying in {delay} seconds...")
            attempts += 1
            time.sleep(delay)
        except Exception as e:
            print(f"Error retrieving sequence for EC {ec} and organism {organism}: {e}")
            return []

    return []

# 读取包含 UniprotID 和 Sequence 列的文件
kcat_df = pd.read_csv("../../Data/database/Kcat_combination_0731_st_2.tsv", sep='\t')

# 筛选出 Sequence 列为空的数据
uniprot_failed_df = kcat_df[kcat_df['Sequence'].isna()].head(50)

# 初始化存储 BRENDA 获取到的序列的字典
brenda_sequences = []

# 使用 BRENDA API 获取序列
email = '1055285901@qq.com'  # 替换为你的 BRENDA 账户邮箱
password = 'LBXSQJLRTZ1124'  # 替换为你的 BRENDA 账户密码

for index, row in uniprot_failed_df.iterrows():
    ec_number = row['ECNumber']
    organism = row['Organism']
    sequences = seq_by_brenda(ec_number, organism, email, password)
    if sequences:
        for sequence in sequences:
            brenda_sequences.append({
                'ECNumber': ec_number,
                'Organism': organism,
                'Smiles': row['Smiles'],
                'Substrate': row['Substrate'],
                'Sequence': sequence,
                'Type': row['EnzymeType'],
                'Value': row['Value'],
            })

# 将获取到的序列数据保存到新的 DataFrame 并展示
brenda_sequences_df = pd.DataFrame(brenda_sequences)
print(brenda_sequences_df)

# 选定展示文件保存路径
output_file_path = "../../Data/database/Kcat_combination_brenda_sequences_sample.tsv"
# 保存数据到新文件
brenda_sequences_df.to_csv(output_file_path, sep='\t', index=False)

print(f"已获取 BRENDA 序列，并保存到新文件 {output_file_path} 中。")


Attempt 1 failed with error: ('Connection broken: IncompleteRead(187385 bytes read, 13657 more expected)', IncompleteRead(187385 bytes read, 13657 more expected)). Retrying in 5 seconds...
Attempt 1 failed with error: ('Connection broken: IncompleteRead(193081 bytes read, 7961 more expected)', IncompleteRead(193081 bytes read, 7961 more expected)). Retrying in 5 seconds...
Attempt 1 failed with error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying in 5 seconds...
Attempt 2 failed with error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying in 5 seconds...
Attempt 1 failed with error: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')). Retrying in 5 seconds...
Attempt 1 failed with error: ('Connection broken: IncompleteRead(161465 bytes read, 39577 more expected)', IncompleteRead(161465 bytes read, 39577 more expected)). Retrying in 5 se